In [6]:
import os 
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [15]:
import osmnx as ox

## IMPORT CITY
city = ox.graph_from_place('Chicago, IL, USA', network_type='drive')

ox.plot_graph(city)

In [21]:
import networkx as nx
import numpy as np

## ADDING EDGE PROPERTIES
city = ox.speed.add_edge_speeds(city)
city = ox.speed.add_edge_travel_times(city)

## GENERATING IMPORTANT INFO
indexes = {node: i for i, node in enumerate(city.nodes)}
travel_times_dict = nx.get_edge_attributes(city, 'travel_time')

## STARTING AND ENDING NODES
str_address = '12949 S Avenue O, Chicago, IL 60633'
end_address = '5033 N Elston Ave, Chicago, IL 60630'

from utils import node_from_address

starting_node = node_from_address(city, str_address)
final_node = node_from_address(city, end_address)

## FILTERING OUT UNDESIRABLE NODES, GENERATING INCIDENCE MATRIX

#banned_coords = [(40.130902, -88.243662), (40.130462, -88.238855), (40.132660, -88.243546)]

#banned_nodes = [ox.nearest_nodes(city, lon, lat) for lat, lon in banned_coords]

banned_nodes= []

edges = list(city.edges)

n_points = len(city.nodes)
n_edges = len(city.edges)

A = np.zeros((n_points, n_edges))

for i, (u, v, k) in enumerate(edges):
    if u in banned_nodes or v in banned_nodes:
        print(u, v)
        continue

    A[indexes[u]][i] = -1
    A[indexes[v]][i] = 1

In [ ]:
## LINEAR PROGRAM LOGIC
import gurobipy as gp
from gurobipy import GRB

b = np.zeros(n_points)

b[indexes[starting_node]] = -1
b[indexes[final_node]] = 1

f = np.zeros(n_edges)

m = gp.Model("lp")

m.Params.LogToConsole = 0
m.Params.Method = 0

f = m.addMVar(shape=n_edges, vtype=GRB.CONTINUOUS, lb=0, name="")

weights = [travel_time for edge, travel_time in travel_times_dict.items()]

m.addConstr(A@f==b)
obj = np.array(weights)@f
m.setObjective(obj, GRB.MINIMIZE)
m.optimize()

flows = m.getAttr("X", m.getVars())

flowed_edges_idx = []
flowed_edges = []

for i in range(len(flows)):
        if flows[i]:
            flowed_edges_idx += [i]

flowed_pts = []

for i, (u, v, k) in enumerate(city.edges):
    if i in flowed_edges_idx:
        flowed_edges += [(u, v)]

        for pt in (u, v):
            if pt not in flowed_pts:
                flowed_pts += [pt]

current_edge = [edge for edge in flowed_edges if edge[0] == starting_node][0]

route_edges = [current_edge]
route_pts = [current_edge[0]]

flowed_edges.remove(current_edge)

while len(flowed_edges):
    for edge in flowed_edges:
        if current_edge[1] == edge[0]:
            route_edges += [edge]
            route_pts += [edge[0]]
            flowed_edges.remove(edge)
            current_edge = edge
            continue

fig, ax = ox.plot_graph_route(city, route_pts, node_color='w', node_edgecolor='k', node_size=0, 
                           node_zorder=3, edge_linewidth=2)

In [20]:
print(b[indexes[final_node]])

1.0


261185552